In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
# Things you should automatically know and have memorized
# N = number of samples
# T = sequence length
# D = number of input features
# M = number of hidden units
# K = number of output units

In [3]:
# Make some data
N = 1
T = 10
D = 3
M = 5
K = 2
X = np.random.randn(N, T, D)

In [4]:
# Set device
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda:0" if torch.cuda.is_available() else "cpu"))
print(device)

mps


In [5]:
# Make an RNN
class SimpleRNN(nn.Module):
  def __init__(self, n_inputs, n_hidden, n_outputs):
    super(SimpleRNN, self).__init__()
    self.D = n_inputs
    self.M = n_hidden
    self.K = n_outputs
    self.rnn = nn.RNN(
        input_size=self.D,
        hidden_size=self.M,
        nonlinearity='tanh',
        batch_first=True)
    self.fc = nn.Linear(self.M, self.K)

  def forward(self, X):
    # initial hidden states
    h0 = torch.zeros(1, X.size(0), self.M).to(device)

    # get RNN unit output
    out, _ = self.rnn(X, h0)

    # we only want h(T) at the final time step
    # out = self.fc(out[:, -1, :])
    out = self.fc(out)
    return out

In [6]:
# Instantiate the model
model = SimpleRNN(n_inputs=D, n_hidden=M, n_outputs=K)
model.to(device)

SimpleRNN(
  (rnn): RNN(3, 5, batch_first=True)
  (fc): Linear(in_features=5, out_features=2, bias=True)
)

In [7]:
# Get the output
inputs = torch.from_numpy(X.astype(np.float32)).to(device)
out = model(inputs)
out

tensor([[[ 1.3262, -0.5604],
         [ 0.9846, -0.2318],
         [ 0.6905, -0.1583],
         [ 1.0501, -0.4207],
         [ 1.0729, -0.4755],
         [ 1.2568, -0.4397],
         [ 0.6949, -0.2753],
         [ 1.4465, -0.4615],
         [ 0.8498, -0.2665],
         [ 1.2232, -0.4880]]], device='mps:0', grad_fn=<LinearBackward0>)

In [8]:
out.shape

torch.Size([1, 10, 2])

In [9]:
# Save for later
Yhats_torch = out.detach().cpu().numpy()

In [10]:
W_xh, W_hh, b_xh, b_hh = model.rnn.parameters()

In [11]:
W_xh.shape

torch.Size([5, 3])

In [12]:
W_xh

Parameter containing:
tensor([[-0.1904,  0.1503,  0.3982],
        [-0.3484, -0.0888, -0.1595],
        [ 0.3781,  0.1323, -0.0995],
        [-0.1574,  0.0477,  0.4104],
        [-0.4359,  0.2843, -0.1153]], device='mps:0', requires_grad=True)

In [13]:
W_xh = W_xh.data.cpu().numpy()
W_xh

array([[-0.19035439,  0.15030614,  0.39815417],
       [-0.34836042, -0.08876104, -0.15953092],
       [ 0.3781055 ,  0.13227014, -0.09947126],
       [-0.15744264,  0.04773243,  0.41038325],
       [-0.43594727,  0.28433383, -0.11526329]], dtype=float32)

In [14]:
b_xh = b_xh.data.cpu().numpy()
W_hh = W_hh.data.cpu().numpy()
b_hh = b_hh.data.cpu().numpy()

In [15]:
# Did we do it right?
W_xh.shape, b_xh.shape, W_hh.shape, b_hh.shape

((5, 3), (5,), (5, 5), (5,))

In [16]:
# Now get the FC layer weights
Wo, bo = model.fc.parameters()

In [17]:
Wo = Wo.data.cpu().numpy()
bo = bo.data.cpu().numpy()
Wo.shape, bo.shape

((2, 5), (2,))

In [18]:
# See if we can replicate the output
h_last = np.zeros(M) # initial hidden state
x = X[0] # the one and only sample
Yhats = np.zeros((T, K)) # where we store the outputs

for t in range(T):
  h = np.tanh(x[t].dot(W_xh.T) + b_xh + h_last.dot(W_hh.T) + b_hh)
  y = h.dot(Wo.T) + bo # we only care about this value on the last iteration
  Yhats[t] = y

  # important: assign h to h_last
  h_last = h

# print the final output
print(Yhats)

[[ 1.3261864  -0.56043035]
 [ 0.98463785 -0.23177617]
 [ 0.69048992 -0.15826304]
 [ 1.05009041 -0.42072841]
 [ 1.07285318 -0.47548414]
 [ 1.25681612 -0.43966157]
 [ 0.69486973 -0.27534767]
 [ 1.4464541  -0.46154145]
 [ 0.84982573 -0.26646923]
 [ 1.22323592 -0.48801956]]


In [19]:
# Check
np.allclose(Yhats, Yhats_torch)

True

In [20]:
# Bonus exercise: calculate the output for multiple samples at once (N > 1)